In [ ]:
import subprocess
import os
import pandas as pd
import numpy as np
import psutil

# Настройки путей
ANSYS_EXE_PATH = r"D:\Program Files\ANSYS Inc\ANSYS Student\v252\ansys\bin\winx64\MAPDL.exe" 
WORKING_DIR = os.getcwd()

Скрипт готов. Рабочая директория: d:\Users\complex_deformations\material_identification


In [ ]:
exp_uni = pd.read_csv('exp_uniaxial.csv')
exp_cyc = pd.read_csv('exp_cyclic.csv')

target_uni = exp_uni['Stress'].values
target_cyc = exp_cyc['Stress'].values

Загружено точек для растяжения: 21
Загружено точек для цикла: 35


In [ ]:
def run_simulation(params, macro_name, output_filename):
    """Вспомогательная функция для запуска одного расчета"""
    # Записываем параметры
    with open('chab_params.txt', 'w') as f:
        for p in params:
            f.write(f"{p}\n")
    
    # Чтобы ANSYS не мешали друг другу, даем разные имена Job-ам
    job_name = f"job_{macro_name}"
    
    # Удаляем старый результат перед запуском
    if os.path.exists(output_filename):
        os.remove(output_filename)

    cmd = [
        ANSYS_EXE_PATH, "-b",
        "-j", job_name,
        "-dir", WORKING_DIR, 
        "-i", f"{macro_name}.mac", 
        "-o", "ansys.out"
    ]

    try:
        subprocess.run(cmd, check=True, capture_output=True)
        # Читаем результат (пропускаем первую нулевую точку [1:])
        df_res = pd.read_csv(output_filename, skipinitialspace=True)
        col_name = df_res.columns[0]
        return df_res[col_name].values[1:]
    except:
        return None

In [ ]:
def objective_function(params):
    # Печать текущих параметров в рантайме
    p_formatted = [f"{p:.2e}" for p in params]
    print(f"Simulating: {p_formatted}")

    # --- ТЕСТ 1: РАСТЯЖЕНИЕ ---
    stress_uni = run_simulation(params, "id_uniaxial", "uniaxial_out.csv")
    if stress_uni is None or len(stress_uni) != len(target_uni):
        print(f"  Fail Uniaxial (Len: {len(stress_uni) if stress_uni is not None else 'None'})")
        return 1e9

    # --- ТЕСТ 2: ЦИКЛ ---
    stress_cyc = run_simulation(params, "id_cyclic", "cyclic_out.csv")
    if stress_cyc is None or len(stress_cyc) != len(target_cyc):
        print(f"  Fail Cyclic (Len: {len(stress_cyc) if stress_cyc is not None else 'None'})")
        return 1e9

    # Расчет ошибок (RMSE)
    rmse_uni = np.sqrt(np.mean((stress_uni - target_uni)**2))
    rmse_cyc = np.sqrt(np.mean((stress_cyc - target_cyc)**2))
    
    # Суммарная ошибка (можно добавить веса, если нужно)
    total_error = 0.4 * rmse_uni + 0.6 * rmse_cyc
    
    print(f"  RMSE Uni: {rmse_uni:.2f} | RMSE Cyc: {rmse_cyc:.2f} | TOTAL: {total_error:.2f}")
    print("-" * 20)
    
    return total_error

In [ ]:
unds = [
    (200, 350),      # sig_y
    (5e4, 5e5),      # c1
    (1000, 10000),   # g1
    (1e4, 5e4),      # c2
    (100, 1000),     # g2
    (500, 5000),     # c3
    (0, 10)          # g3
]

# Не забываем убить старые процессы
def kill_ansys():
    for proc in psutil.process_iter():
        if proc.name() in ['ANSYS.exe', 'MAPDL.exe', 'ansys.exe']:
            proc.kill()

kill_ansys()

result = differential_evolution(
    objective_function, 
    bounds, 
    strategy='best1bin', 
    maxiter=15, 
    popsize=10, 
    disp=True,
    polish=True
)

print("Оптимальные параметры материала найдены:")
print(result.x)

Simulating: [2.2612e+02, 7.1777e+04, 4.3856e+03, 3.2102e+04, 6.6229e+02, 8.3160e+02, 4.1364e+00]
stress_ans_uni is None
Simulating: [2.7311e+02, 1.5134e+05, 2.3455e+03, 1.2035e+04, 8.6402e+02, 3.5194e+03, 4.8100e+01]
